<a href="https://colab.research.google.com/github/gph05010/Daily-Study-Log-TextMining/blob/main/TextMining/ex02_%EC%83%81%ED%92%88_%EB%A6%AC%EB%B7%B0_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EA%B0%90%EC%84%B1_%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# 작업 경로 변경
%cd /content/drive/MyDrive/00 텍스트 마이닝

/content/drive/MyDrive/00 텍스트 마이닝


### 실습 파일 불러오기
- naver_shopping.txt 파일 df로 불러오기 -> data
- \t 으로 데이터 구분 되어있음

In [3]:
# 필요한 라이브러리 import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:
# 데이터 불러오기
data = pd.read_csv("./data/naver_shopping.txt", delimiter="\t", header=None)
# read_scv 함수는 맨 첫 번째 줄을 header로 인식하여 컬럼 이름으로 넣기 때문에, header가 파일에 없다는 것을 알려주어야 함

In [7]:
data

,0,1
0,5,배공빠르고 굿
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ
...,...,...
199995,2,장마라그런가!!! 달지않아요
199996,5,다이슨 케이스 구매했어요 다이슨 슈퍼소닉 드라이기 케이스 구매했어요가격 괜찮고 배송...
199997,5,로드샾에서 사는것보다 세배 저렴하네요 ㅜㅜ 자주이용할께요
199998,5,넘이쁘고 쎄련되보이네요~


In [8]:
# 컬럼명 업데이트
data.columns = ["평점", "리뷰"]
data.head()

,평점,리뷰
0,5,배공빠르고 굿
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ


In [9]:
# 필요한 리뷰데이터만 인덱싱
text = data["리뷰"]

### 텍스트 데이터 전처리
1. 결측치 확인
2. 특수문자 제거

In [10]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   평점      200000 non-null  int64 
 1   리뷰      200000 non-null  object
dtypes: int64(1), object(1)
memory usage: 3.1+ MB


In [11]:
# re 모듈을 활용해 특수문자를 제거
import re # 정규 표현식을 사용하기 위한 모듈

| 특수 문자        | 설명                                                                 |
|------------------|----------------------------------------------------------------------|
| `.`              | 한 개의 임의의 문자 (단, 줄바꿈 문자 `\n` 제외)                       |
| `?`              | 앞의 문자가 **있을 수도, 없을 수도** 있음 (0개 또는 1개)             |
| `*`              | 앞의 문자가 **0개 이상** 반복됨                                       |
| `+`              | 앞의 문자가 **1개 이상** 반복됨                                       |
| `^`              | **문자열의 시작**을 의미                                              |
| `$`              | **문자열의 끝**을 의미                                                |
| `{숫자}`         | 정확히 해당 **숫자만큼 반복**함                                       |
| `{숫자1, 숫자2}` | **숫자1 이상 숫자2 이하** 만큼 반복함                                 |
| `{숫자,}`        | **숫자 이상** 반복함                                                  |
| `[...]`          | 대괄호 안의 문자들 중 **하나**와 매치. 예: `[abc]` → a 또는 b 또는 c |
| `[a-z]`          | **소문자 알파벳 범위**와 매치                                         |
| `[a-zA-Z]`       | **모든 알파벳**과 매치                                                |
| `[^문자]`        | **해당 문자를 제외한 문자**와 매치                                    |


In [13]:
# 전화번호 검출
# 추출하고 싶은 전화번호의 형태를 생각해보자!
# 010-1234-5678, 01012345678, 010-12345678
# 중간에 하이픈이 있거나 없거나 하는 모든 패턴을 검출해보자!

# 정규 표현식을 사용해 규칙 생성
p = re.compile("010-?[0-9]{4}-?[0-9]{4}")
# ? : 앞에 있는 문자가 있을 수도 없을 수도 있다!(1 or 0개)
# [] : 대괄호 안의 문자 하나와 매칭
# {숫자} : 앞에 있는 문자가 숫자만큼 반복

In [16]:
# 생성한 규칙을 가지고 문자 패턴 검색
result = p.search("내 전화번호는 010-1234-5678이야!")
# match : 정해진 규칙과 일치하는 부분
# span = (시작점, 끝점 +1)
str1 = "내 전화번호는 010-1234-5678이야!"
str1[8:21]

'010-1234-5678'

In [18]:
result.start
result.end
# 바로 슬라이싱에 활용 가능

<function Match.end(group=0, /)>

In [19]:
# 리뷰 데이터에 특수문자 제거!
# 패턴 저장
pattern = r"[^a-zA-z0-9가-힣\s\.\?\!]"

# [^] : not
# a-z : 영어 소문자
# A-z : 영어 대문자
# 0-9 : 숫자
# 가-힣 : 한글
# \s : 띄어쓰기, \. : 온점, \? : 물음표, \! : 느낌표


In [20]:
# 반복문을 통해 데이터에 모두 패턴 적용
from tqdm import tqdm

new_doc = []
for doc in tqdm(text) :
  clean_doc = re.sub(pattern, "", doc) # 해당 패턴에 해당하는 부분을 삭제
  new_doc.append(clean_doc)

100%|██████████| 200000/200000 [00:00<00:00, 279849.74it/s]


In [21]:
# 정제된 텍스트 데이터를 새로운 컬럼으로 추가
data["cleaned_doc"] = new_doc

In [22]:
data.head()
# 감성 분석인데 ㅎㅎ를 왜 삭제하는 거지...?

,평점,리뷰,cleaned_doc
0,5,배공빠르고 굿,배공빠르고 굿
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요


### 이모지 제거

In [23]:
# 필요한 패키지 설치
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 26.4 MB/s eta 0:00:00


In [24]:
temp = 'DM으로 연락바랍니다 🙏🙏 😊🤣😍'

In [25]:
import emoji

In [26]:
# 이모지 -> 텍스트 변환
emoji.demojize(temp)
# 이모지를 해석해서 문자열로 변환

'DM으로 연락바랍니다 :folded_hands::folded_hands: :smiling_face_with_smiling_eyes::rolling_on_the_floor_laughing::smiling_face_with_heart-eyes:'

In [28]:
# 이모지 삭제
emoji.replace_emoji(temp).strip() # strip() : 뒤에 남은 띄어쓰기 제거

'DM으로 연락바랍니다'